In [35]:
import numpy as np

def generate_arrivals(max_time=365):
    """
    Pre-generates all arrivals and lengths of stay (LOS) for a single year.
    Returns a list of sorted tuples: (arrival_time, patient_type, length_of_stay)
    """
    arrivals = []
    
    # Ward A (Regular): lambda_1(t) = -(1/3650)t^2 + (1/10)t
    t = 0
    max_lambda_A = 9.125 
    while True:
        t -= (1 / max_lambda_A) * np.log(np.random.rand())
        if t > max_time: break
        lambda_t = -(1/3650) * t**2 + 0.1 * t
        if np.random.rand() <= (lambda_t / max_lambda_A):
            sigma = np.sqrt(np.log(2))
            los = np.random.lognormal(mean=np.log(4*np.sqrt(2)), sigma=sigma)
            arrivals.append((t, 'A', los))

    # Ward B (Intensive): lambda_2(t) = 0.2 * lambda_1(t)
    t = 0
    max_lambda_B = 1.825 
    while True:
        t -= (1 / max_lambda_B) * np.log(np.random.rand())
        if t > max_time: break
        lambda_t = 0.2 * (-(1/3650) * t**2 + 0.1 * t)
        if np.random.rand() <= (lambda_t / max_lambda_B):
            sigma = np.sqrt(np.log(2))
            los = np.random.lognormal(mean=np.log(6*np.sqrt(2)), sigma=sigma)
            arrivals.append((t, 'B', los))

    # Ward C (Other): Homogeneous Poisson Process, lambda_3 = 6
    t = 0
    lambda_C = 6
    while True:
        t -= (1 / lambda_C) * np.log(np.random.rand())
        if t > max_time: break
        sigma = np.sqrt(np.log(2))
        los = np.random.lognormal(mean=np.log(5*np.sqrt(2)), sigma=sigma)
        arrivals.append((t, 'C', los))

    # Sort all arrivals chronologically
    arrivals.sort(key=lambda x: x[0])
    return arrivals

n_replications = 5
replications_data = []

for i in range(n_replications):
    # Set a unique seed per replication to make them reproducible
    np.random.seed(42 + i)
    replications_data.append(generate_arrivals(max_time=365))

print(f"Pre-generated arrivals and stays for {n_replications} replications.")

Pre-generated arrivals and stays for 5 replications.


In [39]:
import numpy as np

def generate_arrivals_discrete(max_time=365):
    """
    Generates arrivals day-by-day using Poisson distributions.
    Returns a list of sorted tuples: (arrival_time, patient_type, length_of_stay)
    """
    arrivals = []
    sigma = np.sqrt(np.log(2))
    
    for d in range(max_time+1):
        # We evaluate the rate at the midpoint of the day (d + 0.5)
        t_mid = d + 0.5
        
        # --- 1. Ward A (Regular) ---
        # Calculate rate, ensuring it doesn't go negative
        lambda_A = max(0, -(1/3650) * (t_mid**2) + 0.1 * t_mid)
        # Sample the number of arrivals for this day
        n_A = np.random.poisson(lambda_A)
        
        for _ in range(n_A):
            # Distribute the arrival time uniformly within day [d, d+1]
            arrival_time = d + np.random.rand()
            los = np.random.lognormal(mean=np.log(4*np.sqrt(2)), sigma=sigma)
            arrivals.append((arrival_time, 'A', los))
            
        # --- 2. Ward B (Intensive) ---
        lambda_B = 0.2 * lambda_A
        n_B = np.random.poisson(lambda_B)
        for _ in range(n_B):
            arrival_time = d + np.random.rand()
            los = np.random.lognormal(mean=np.log(6*np.sqrt(2)), sigma=sigma)
            arrivals.append((arrival_time, 'B', los))
            
        # --- 3. Ward C (Other) ---
        lambda_C = 6.0
        n_C = np.random.poisson(lambda_C)
        for _ in range(n_C):
            arrival_time = d + np.random.rand()
            los = np.random.lognormal(mean=np.log(5*np.sqrt(2)), sigma=sigma)
            arrivals.append((arrival_time, 'C', los))
            
    # Sort all arrivals chronologically so the simulation can process them in order
    arrivals.sort(key=lambda x: x[0])
    return arrivals



n_replications = 5
replications_data = []

for i in range(n_replications):
    # Set a unique seed per replication to make them reproducible
    np.random.seed(42 + i)
    replications_data.append(generate_arrivals_discrete(max_time=365))

print(f"Pre-generated arrivals and stays for {n_replications} replications.")

Pre-generated arrivals and stays for 5 replications.


In [40]:
def simulate_hospital_fast(cap_A, cap_B, cap_C, arrivals, max_time=365):
    # Lists to store the departure times of patients currently in beds
    beds_A = []
    beds_B = []
    beds_C = []
    
    arrivals_count = {'A': 0, 'B': 0, 'C': 0}
    relocated = {'A': 0, 'B': 0, 'C': 0}
    util_integral = {'A': 0.0, 'B': 0.0, 'C': 0.0}
    
    for t, p_type, los in arrivals:
        # 1. Update occupancy: remove patients who have already departed before time t
        beds_A = [dep for dep in beds_A if dep > t]
        beds_B = [dep for dep in beds_B if dep > t]
        beds_C = [dep for dep in beds_C if dep > t]
        
        # 2. Process the arrival
        arrivals_count[p_type] += 1
        departure_time = t + los
        
        if p_type == 'C':
            if len(beds_C) < cap_C:
                beds_C.append(departure_time)
                # Option A: util_integral['C'] += los  (Matches original code exactly)
                # Option B: util_integral['C'] += min(departure_time, max_time) - t (Capped at max_time)
                util_integral['C'] += los
            else:
                relocated['C'] += 1
                
        elif p_type == 'B':
            if len(beds_B) < cap_B:
                beds_B.append(departure_time)
                util_integral['B'] += los
            elif len(beds_A) < cap_A: # Overflow to Ward A
                beds_A.append(departure_time)
                util_integral['A'] += los
            else:
                relocated['B'] += 1
                
        elif p_type == 'A':
            if len(beds_A) < cap_A:
                beds_A.append(departure_time)
                util_integral['A'] += los
            else:
                relocated['A'] += 1
                
    # Calculate final results
    blocking_prob = {
        k: (relocated[k] / arrivals_count[k] if arrivals_count[k] > 0 else 0) 
        for k in ['A', 'B', 'C']
    }
    
    avg_utilization = {
        'A': util_integral['A'] / (cap_A * max_time) if cap_A > 0 else 0,
        'B': util_integral['B'] / (cap_B * max_time) if cap_B > 0 else 0,
        'C': util_integral['C'] / (cap_C * max_time) if cap_C > 0 else 0
    }
    
    total_relocated = sum(relocated.values())
    
    return {
        "arrivals": arrivals_count,
        "relocated": relocated,
        "blocking_probabilities": blocking_prob,
        "total_relocated": total_relocated,
        "avg_utilization": avg_utilization
    }

In [42]:
from tqdm.notebook import tqdm

total_beds = 75
min_beds = 1

best_allocation = None
min_avg_relocated = float('inf')

print(f"Starting grid search. Evaluating combinations for {total_beds} beds...")
print(f"Running {n_replications} replications per combination.\n")

# Wrap the outer loop with tqdm to display a progress bar
for cap_A in tqdm(range(min_beds, total_beds - 2*min_beds + 1), desc="Grid Search Progress"):
    for cap_B in range(min_beds, total_beds - cap_A - min_beds + 1):
        cap_C = total_beds - cap_A - cap_B
        
        relocated_counts = []
        # Reuse pre-generated replications (Common Random Numbers)
        for rep_arrivals in replications_data:
            res = simulate_hospital_fast(cap_A, cap_B, cap_C, rep_arrivals, max_time=365)
            relocated_counts.append(res['total_relocated'])
            
        avg_relocated = np.mean(relocated_counts)
        
        if avg_relocated < min_avg_relocated:
            min_avg_relocated = avg_relocated
            best_allocation = (cap_A, cap_B, cap_C)
            
            # Print updates on new bests alongside the progress bar
            print(f"New Best Found -> Ward A: {cap_A:2d} | Ward B: {cap_B:2d} | Ward C: {cap_C:2d} " 
                  f"| Expected Relocations: {min_avg_relocated:.2f}")

Starting grid search. Evaluating combinations for 75 beds...
Running 5 replications per combination.



Grid Search Progress:   0%|          | 0/73 [00:00<?, ?it/s]

New Best Found -> Ward A:  1 | Ward B:  1 | Ward C: 73 | Expected Relocations: 2588.80
New Best Found -> Ward A:  1 | Ward B:  2 | Ward C: 72 | Expected Relocations: 2567.60
New Best Found -> Ward A:  1 | Ward B:  3 | Ward C: 71 | Expected Relocations: 2547.80
New Best Found -> Ward A:  1 | Ward B:  4 | Ward C: 70 | Expected Relocations: 2530.80
New Best Found -> Ward A:  1 | Ward B:  5 | Ward C: 69 | Expected Relocations: 2513.40
New Best Found -> Ward A:  1 | Ward B:  6 | Ward C: 68 | Expected Relocations: 2505.40
New Best Found -> Ward A:  1 | Ward B:  7 | Ward C: 67 | Expected Relocations: 2490.40
New Best Found -> Ward A:  1 | Ward B:  8 | Ward C: 66 | Expected Relocations: 2476.20
New Best Found -> Ward A:  1 | Ward B:  9 | Ward C: 65 | Expected Relocations: 2470.60
New Best Found -> Ward A:  1 | Ward B: 10 | Ward C: 64 | Expected Relocations: 2469.60
New Best Found -> Ward A:  1 | Ward B: 11 | Ward C: 63 | Expected Relocations: 2464.00
New Best Found -> Ward A:  2 | Ward B:  7 |